# Лабораторная работа № 3
## Профилирование и качество данных: переход от raw к staging

**Датасет:** Bike Sharing Dataset, UCI  
**Входной файл:** `hour.csv`  
**Логическая дата загрузки raw:** `2026-10-18`

Контур этой работы: `raw → staging`.


## Контекст из Project Brief

**Основной аналитический вопрос:** как различается количество аренд велосипедов в зависимости от сезона и погодных условий?

**Единица анализа:** одно почасовое наблюдение.

**Бизнес-ключ:** `dteday + hr` — дата и час наблюдения.

Для профилирования используются поля `dteday`, `hr`, `season`, `weathersit`, `cnt`.  
Поля `casual` и `registered` используются для проверки согласованности с `cnt`.


In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
ENV_PATH = PROJECT_ROOT / ".env"

def load_simple_env(path):
    values = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8-sig").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip().strip('"').strip("'")
    return values

env = load_simple_env(ENV_PATH)

S3_ACCESS_KEY = env.get("S3_ACCESS_KEY", "s3admin")
S3_SECRET_KEY = env.get("S3_SECRET_KEY", "s3admin123")
S3_ENDPOINT = env.get("S3_ENDPOINT", "localhost:8333")

DATASET_SLUG = "bike-sharing"
INGESTED_ON = "2026-10-18"

SOURCE_PATH = (
    f"s3://raw/{DATASET_SLUG}/ingested_on={INGESTED_ON}/hour.csv"
)

STAGING_PATH = (
    f"s3://staging/{DATASET_SLUG}/"
    f"source_ingested_on={INGESTED_ON}/staging.parquet"
)

QUARANTINE_PATH = (
    f"s3://staging/{DATASET_SLUG}/"
    f"source_ingested_on={INGESTED_ON}/quarantine.parquet"
)

THRESHOLD_VIOLATIONS = 0
THRESHOLD_SHARE = 0.0

print("SOURCE_PATH:", SOURCE_PATH)
print("STAGING_PATH:", STAGING_PATH)
print("QUARANTINE_PATH:", QUARANTINE_PATH)


SOURCE_PATH: s3://raw/bike-sharing/ingested_on=2026-10-18/hour.csv
STAGING_PATH: s3://staging/bike-sharing/source_ingested_on=2026-10-18/staging.parquet
QUARANTINE_PATH: s3://staging/bike-sharing/source_ingested_on=2026-10-18/quarantine.parquet


In [2]:
con = duckdb.connect()

try:
    con.execute("LOAD httpfs")
except Exception:
    con.execute("INSTALL httpfs")
    con.execute("LOAD httpfs")

con.execute(f"SET s3_endpoint='{S3_ENDPOINT}'")
con.execute(f"SET s3_access_key_id='{S3_ACCESS_KEY}'")
con.execute(f"SET s3_secret_access_key='{S3_SECRET_KEY}'")
con.execute("SET s3_region='us-east-1'")
con.execute("SET s3_url_style='path'")
con.execute("SET s3_use_ssl=false")

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW raw_data AS
    SELECT * FROM read_csv_auto('{SOURCE_PATH}')
    """
)

print("DuckDB:", duckdb.__version__)
print("Чтение raw через S3 выполнено.")

DuckDB: 1.1.3
Чтение raw через S3 выполнено.


## 1. Чтение и общий профиль

По заданию нужно зафиксировать число строк и полей, фактическую схему выбранных полей, несколько исходных строк, пропуски, повторы по бизнес-ключу, период и распределение по часам.


In [3]:
rows_total = int(
    con.execute("SELECT COUNT(*) FROM raw_data").fetchone()[0]
)
columns_total = len(con.execute("DESCRIBE raw_data").fetchall())

pd.DataFrame([{
    "rows_total": rows_total,
    "columns_total": columns_total,
}])


,rows_total,columns_total
0,17379,18


In [4]:
selected_fields = [
    "dteday",
    "hr",
    "season",
    "weathersit",
    "cnt",
    "casual",
    "registered",
]

schema = con.execute("DESCRIBE raw_data").df()
schema


,column_name,column_type,null,key,default,extra
0,instant,BIGINT,YES,None,None,None
1,dteday,DATE,YES,None,None,None
2,season,BIGINT,YES,None,None,None
3,yr,BIGINT,YES,None,None,None
4,mnth,BIGINT,YES,None,None,None
5,hr,BIGINT,YES,None,None,None
6,holiday,BIGINT,YES,None,None,None
7,weekday,BIGINT,YES,None,None,None
8,workingday,BIGINT,YES,None,None,None
9,weathersit,BIGINT,YES,None,None,None


In [5]:
con.execute(
    "SELECT dteday, hr, season, weathersit, cnt, casual, registered "
    "FROM raw_data LIMIT 5"
).df()


,dteday,hr,season,weathersit,cnt,casual,registered
0,2011-01-01,0,1,1,16,3,13
1,2011-01-01,1,1,1,40,8,32
2,2011-01-01,2,1,1,32,5,27
3,2011-01-01,3,1,1,13,3,10
4,2011-01-01,4,1,1,1,0,1


In [6]:
missing_profile = con.execute(
    "SELECT 'dteday' AS field, COUNT(*) AS rows_checked, "
    "COUNT(*) FILTER (WHERE dteday IS NULL) AS missing_rows, "
    "COUNT(*) FILTER (WHERE dteday IS NULL) * 1.0 / COUNT(*) AS missing_share "
    "FROM raw_data "
    "UNION ALL "
    "SELECT 'hr', COUNT(*), COUNT(*) FILTER (WHERE hr IS NULL), "
    "COUNT(*) FILTER (WHERE hr IS NULL) * 1.0 / COUNT(*) FROM raw_data "
    "UNION ALL "
    "SELECT 'season', COUNT(*), COUNT(*) FILTER (WHERE season IS NULL), "
    "COUNT(*) FILTER (WHERE season IS NULL) * 1.0 / COUNT(*) FROM raw_data "
    "UNION ALL "
    "SELECT 'weathersit', COUNT(*), COUNT(*) FILTER (WHERE weathersit IS NULL), "
    "COUNT(*) FILTER (WHERE weathersit IS NULL) * 1.0 / COUNT(*) FROM raw_data "
    "UNION ALL "
    "SELECT 'cnt', COUNT(*), COUNT(*) FILTER (WHERE cnt IS NULL), "
    "COUNT(*) FILTER (WHERE cnt IS NULL) * 1.0 / COUNT(*) FROM raw_data"
).df()

missing_profile


,field,rows_checked,missing_rows,missing_share
0,dteday,17379,0,0.0
1,hr,17379,0,0.0
2,season,17379,0,0.0
3,weathersit,17379,0,0.0
4,cnt,17379,0,0.0


In [7]:
duplicate_profile = con.execute(
    "SELECT COUNT(*) AS duplicate_groups, "
    "COALESCE(SUM(group_rows - 1), 0) AS duplicate_extra_rows "
    "FROM ("
    "SELECT dteday, hr, COUNT(*) AS group_rows "
    "FROM raw_data "
    "GROUP BY dteday, hr "
    "HAVING COUNT(*) > 1"
    ")"
).df()

duplicate_profile


,duplicate_groups,duplicate_extra_rows
0,0,0.0


In [8]:
con.execute(
    "SELECT "
    "MIN(TRY_CAST(dteday AS DATE)) AS min_date, "
    "MAX(TRY_CAST(dteday AS DATE)) AS max_date "
    "FROM raw_data"
).df()


,min_date,max_date
0,2011-01-01,2012-12-31


In [9]:
hour_distribution = con.execute(
    "SELECT hr, COUNT(*) AS rows_count, "
    "ROUND(AVG(cnt), 2) AS avg_rentals, "
    "MIN(cnt) AS min_rentals, MAX(cnt) AS max_rentals "
    "FROM raw_data "
    "GROUP BY hr "
    "ORDER BY hr"
).df()

hour_distribution


,hr,rows_count,avg_rentals,min_rentals,max_rentals
0,0,726,53.90,2,283
1,1,724,33.38,1,168
2,2,715,22.87,1,132
3,3,697,11.73,1,79
4,4,697,6.35,1,28
5,5,717,19.89,1,66
6,6,725,76.04,1,213
7,7,727,212.06,1,596
8,8,727,359.01,5,839
9,9,727,219.31,14,426


### Наблюдения по профилю

1. В выбранных значимых полях `dteday`, `hr`, `season`, `weathersit` и `cnt` пропуски отсутствуют: проверено 17 379 строк, доля пропусков равна 0%. Это важно, поскольку выбранные поля используются для временных, сезонных и погодных срезов.

2. Спрос заметно зависит от часа суток: среднее количество аренд минимально в 04:00 (6,35), а максимально в 17:00 (461,45). Это важно для основного аналитического вопроса, поскольку внутридневной профиль спроса может различаться при разных сезонных и погодных условиях.

## 2. Правила качества

Выбрано семь правил. Порог количества нарушений для всех правил равен `0`, порог доли — `0.0`. Порог задаётся переменными и используется в запросах.

Критические нарушения (`critical`) отправляются в карантин. Предупреждающие (`warning`) остаются в `staging` и получают явный признак.


### UQ01 — уникальность бизнес-ключа

**Поля:** `dteday + hr`  
**Проверяемые строки:** все  
**Условие:** один ключ должен соответствовать не более чем одной записи  
**Порог:** 0 нарушений  
**Критичность:** `critical`  
**Действие:** `quarantine`  
**Связь с задачей:** дубликаты искажают почасовые сравнения спроса.


In [10]:
rule_uq01 = con.execute(
    f"WITH x AS ("
    f"SELECT *, COUNT(*) OVER (PARTITION BY dteday, hr) AS key_count "
    f"FROM raw_data) "
    f"SELECT COUNT(*) AS rows_checked, "
    f"COUNT(*) FILTER ("
    f"WHERE dteday IS NOT NULL AND hr IS NOT NULL AND key_count > 1"
    f") AS violations, "
    f"COUNT(*) FILTER ("
    f"WHERE dteday IS NOT NULL AND hr IS NOT NULL AND key_count > 1"
    f") * 1.0 / COUNT(*) AS violation_share, "
    f"{THRESHOLD_VIOLATIONS} AS threshold, "
    f"CASE WHEN COUNT(*) FILTER ("
    f"WHERE dteday IS NOT NULL AND hr IS NOT NULL AND key_count > 1"
    f") <= {THRESHOLD_VIOLATIONS} "
    f"THEN 'PASS' ELSE 'FAIL' END AS result "
    f"FROM x"
).df()

rule_uq01


,rows_checked,violations,violation_share,threshold,result
0,17379,0,0.0,0,PASS


### CM01 — полнота значимых полей

**Поля:** `dteday`, `hr`, `season`, `weathersit`, `cnt`  
**Проверяемые строки:** все  
**Условие:** ни одно значимое поле не должно быть пропущено  
**Порог:** 0 нарушений  
**Критичность:** `critical`  
**Действие:** `quarantine`


In [11]:
rule_cm01 = con.execute(
    f"SELECT COUNT(*) AS rows_checked, "
    f"COUNT(*) FILTER ("
    f"WHERE dteday IS NULL OR hr IS NULL OR season IS NULL "
    f"OR weathersit IS NULL OR cnt IS NULL"
    f") AS violations, "
    f"COUNT(*) FILTER ("
    f"WHERE dteday IS NULL OR hr IS NULL OR season IS NULL "
    f"OR weathersit IS NULL OR cnt IS NULL"
    f") * 1.0 / COUNT(*) AS violation_share, "
    f"{THRESHOLD_VIOLATIONS} AS threshold, "
    f"CASE WHEN COUNT(*) FILTER ("
    f"WHERE dteday IS NULL OR hr IS NULL OR season IS NULL "
    f"OR weathersit IS NULL OR cnt IS NULL"
    f") <= {THRESHOLD_VIOLATIONS} "
    f"THEN 'PASS' ELSE 'FAIL' END AS result "
    f"FROM raw_data"
).df()

rule_cm01


,rows_checked,violations,violation_share,threshold,result
0,17379,0,0.0,0,PASS


### RG01 — неотрицательное количество аренд

**Поле:** `cnt`  
**Условие:** `cnt >= 0`  
**Порог:** 0 нарушений  
**Критичность:** `critical`  
**Действие:** `quarantine`  
**Связь с задачей:** `cnt` — основная метрика спроса.


In [12]:
rule_rg01 = con.execute(
    f"SELECT COUNT(*) FILTER (WHERE cnt IS NOT NULL) AS rows_checked, "
    f"COUNT(*) FILTER (WHERE cnt IS NOT NULL AND cnt < 0) AS violations, "
    f"COUNT(*) FILTER (WHERE cnt IS NOT NULL AND cnt < 0) * 1.0 / "
    f"NULLIF(COUNT(*) FILTER (WHERE cnt IS NOT NULL), 0) AS violation_share, "
    f"{THRESHOLD_VIOLATIONS} AS threshold, "
    f"CASE WHEN COUNT(*) FILTER (WHERE cnt IS NOT NULL AND cnt < 0) "
    f"<= {THRESHOLD_VIOLATIONS} THEN 'PASS' ELSE 'FAIL' END AS result "
    f"FROM raw_data"
).df()

rule_rg01


,rows_checked,violations,violation_share,threshold,result
0,17379,0,0.0,0,PASS


### RG02 — допустимый диапазон часа

**Поле:** `hr`  
**Условие:** `0 <= hr <= 23`  
**Порог:** 0 нарушений  
**Критичность:** `critical`  
**Действие:** `quarantine`  
**Связь с задачей:** ошибочный час искажает внутридневной анализ спроса.


In [13]:
rule_rg02 = con.execute(
    f"SELECT COUNT(*) FILTER (WHERE hr IS NOT NULL) AS rows_checked, "
    f"COUNT(*) FILTER (WHERE hr IS NOT NULL AND (hr < 0 OR hr > 23)) AS violations, "
    f"COUNT(*) FILTER (WHERE hr IS NOT NULL AND (hr < 0 OR hr > 23)) * 1.0 / "
    f"NULLIF(COUNT(*) FILTER (WHERE hr IS NOT NULL), 0) AS violation_share, "
    f"{THRESHOLD_VIOLATIONS} AS threshold, "
    f"CASE WHEN COUNT(*) FILTER (WHERE hr IS NOT NULL AND (hr < 0 OR hr > 23)) "
    f"<= {THRESHOLD_VIOLATIONS} THEN 'PASS' ELSE 'FAIL' END AS result "
    f"FROM raw_data"
).df()

rule_rg02


,rows_checked,violations,violation_share,threshold,result
0,17379,0,0.0,0,PASS


### CAT01 — допустимый код сезона

**Поле:** `season`  
**Условие:** значение принадлежит `{1, 2, 3, 4}`  
**Порог:** 0 нарушений  
**Критичность:** `warning`  
**Действие:** `flag`  
**Связь с задачей:** некорректные коды нарушают сравнение сезонов.


In [14]:
rule_cat01 = con.execute(
    f"SELECT COUNT(*) FILTER (WHERE season IS NOT NULL) AS rows_checked, "
    f"COUNT(*) FILTER (WHERE season IS NOT NULL AND season NOT IN (1,2,3,4)) AS violations, "
    f"COUNT(*) FILTER (WHERE season IS NOT NULL AND season NOT IN (1,2,3,4)) * 1.0 / "
    f"NULLIF(COUNT(*) FILTER (WHERE season IS NOT NULL), 0) AS violation_share, "
    f"{THRESHOLD_VIOLATIONS} AS threshold, "
    f"CASE WHEN COUNT(*) FILTER ("
    f"WHERE season IS NOT NULL AND season NOT IN (1,2,3,4)"
    f") <= {THRESHOLD_VIOLATIONS} THEN 'PASS' ELSE 'FAIL' END AS result "
    f"FROM raw_data"
).df()

rule_cat01


,rows_checked,violations,violation_share,threshold,result
0,17379,0,0.0,0,PASS


### CAT02 — допустимый код погодной ситуации

**Поле:** `weathersit`  
**Условие:** значение принадлежит `{1, 2, 3, 4}`  
**Порог:** 0 нарушений  
**Критичность:** `warning`  
**Действие:** `flag`  
**Связь с задачей:** некорректные коды нарушают сравнение погодных условий.


In [15]:
rule_cat02 = con.execute(
    f"SELECT COUNT(*) FILTER (WHERE weathersit IS NOT NULL) AS rows_checked, "
    f"COUNT(*) FILTER ("
    f"WHERE weathersit IS NOT NULL AND weathersit NOT IN (1,2,3,4)"
    f") AS violations, "
    f"COUNT(*) FILTER ("
    f"WHERE weathersit IS NOT NULL AND weathersit NOT IN (1,2,3,4)"
    f") * 1.0 / "
    f"NULLIF(COUNT(*) FILTER (WHERE weathersit IS NOT NULL), 0) AS violation_share, "
    f"{THRESHOLD_VIOLATIONS} AS threshold, "
    f"CASE WHEN COUNT(*) FILTER ("
    f"WHERE weathersit IS NOT NULL AND weathersit NOT IN (1,2,3,4)"
    f") <= {THRESHOLD_VIOLATIONS} THEN 'PASS' ELSE 'FAIL' END AS result "
    f"FROM raw_data"
).df()

rule_cat02


,rows_checked,violations,violation_share,threshold,result
0,17379,0,0.0,0,PASS


### CONS01 — согласованность `casual + registered = cnt`

**Поля:** `casual`, `registered`, `cnt`  
**Условие:** сумма двух компонентов должна быть равна общей метрике  
**Порог:** 0 нарушений  
**Критичность:** `critical`  
**Действие:** `quarantine`  
**Связь с задачей:** `cnt` должен согласовываться с компонентами общего спроса.


In [16]:
rule_cons01 = con.execute(
    f"SELECT COUNT(*) FILTER ("
    f"WHERE casual IS NOT NULL AND registered IS NOT NULL AND cnt IS NOT NULL"
    f") AS rows_checked, "
    f"COUNT(*) FILTER ("
    f"WHERE casual IS NOT NULL AND registered IS NOT NULL AND cnt IS NOT NULL "
    f"AND casual + registered <> cnt"
    f") AS violations, "
    f"COUNT(*) FILTER ("
    f"WHERE casual IS NOT NULL AND registered IS NOT NULL AND cnt IS NOT NULL "
    f"AND casual + registered <> cnt"
    f") * 1.0 / NULLIF(COUNT(*) FILTER ("
    f"WHERE casual IS NOT NULL AND registered IS NOT NULL AND cnt IS NOT NULL"
    f"), 0) AS violation_share, "
    f"{THRESHOLD_VIOLATIONS} AS threshold, "
    f"CASE WHEN COUNT(*) FILTER ("
    f"WHERE casual IS NOT NULL AND registered IS NOT NULL AND cnt IS NOT NULL "
    f"AND casual + registered <> cnt"
    f") <= {THRESHOLD_VIOLATIONS} "
    f"THEN 'PASS' ELSE 'FAIL' END AS result "
    f"FROM raw_data"
).df()

rule_cons01


,rows_checked,violations,violation_share,threshold,result
0,17379,0,0.0,0,PASS


In [17]:
rules_summary = pd.concat([
    rule_uq01.assign(rule_id="UQ01", criticality="critical", action="quarantine"),
    rule_cm01.assign(rule_id="CM01", criticality="critical", action="quarantine"),
    rule_rg01.assign(rule_id="RG01", criticality="critical", action="quarantine"),
    rule_rg02.assign(rule_id="RG02", criticality="critical", action="quarantine"),
    rule_cat01.assign(rule_id="CAT01", criticality="warning", action="flag"),
    rule_cat02.assign(rule_id="CAT02", criticality="warning", action="flag"),
    rule_cons01.assign(rule_id="CONS01", criticality="critical", action="quarantine"),
], ignore_index=True)[[
    "rule_id",
    "rows_checked",
    "violations",
    "violation_share",
    "threshold",
    "result",
    "criticality",
    "action",
]]

rules_summary


,rule_id,rows_checked,violations,violation_share,threshold,result,criticality,action
0,UQ01,17379,0,0.0,0,PASS,critical,quarantine
1,CM01,17379,0,0.0,0,PASS,critical,quarantine
2,RG01,17379,0,0.0,0,PASS,critical,quarantine
3,RG02,17379,0,0.0,0,PASS,critical,quarantine
4,CAT01,17379,0,0.0,0,PASS,warning,flag
5,CAT02,17379,0,0.0,0,PASS,warning,flag
6,CONS01,17379,0,0.0,0,PASS,critical,quarantine


## 3. Нормализация, `staging` и карантин

Исходные данные из `raw` не изменяются. Типы аналитических полей приводятся явно. Критические нарушения попадают в `quarantine`, а предупреждающие нарушения сохраняются в `staging` с признаками качества.


In [18]:
con.execute(
    "CREATE OR REPLACE TEMP VIEW normalized_with_quality AS "
    "WITH base AS ("
    "SELECT "
    "CAST(instant AS INTEGER) AS instant, "
    "TRY_CAST(dteday AS DATE) AS dteday, "
    "CAST(season AS INTEGER) AS season, "
    "CAST(yr AS INTEGER) AS yr, "
    "CAST(mnth AS INTEGER) AS mnth, "
    "CAST(hr AS INTEGER) AS hr, "
    "CAST(holiday AS INTEGER) AS holiday, "
    "CAST(weekday AS INTEGER) AS weekday, "
    "CAST(workingday AS INTEGER) AS workingday, "
    "CAST(weathersit AS INTEGER) AS weathersit, "
    "CAST(temp AS DOUBLE) AS temp, "
    "CAST(atemp AS DOUBLE) AS atemp, "
    "CAST(hum AS DOUBLE) AS hum, "
    "CAST(windspeed AS DOUBLE) AS windspeed, "
    "CAST(casual AS INTEGER) AS casual, "
    "CAST(registered AS INTEGER) AS registered, "
    "CAST(cnt AS INTEGER) AS cnt, "
    "COUNT(*) OVER (PARTITION BY TRY_CAST(dteday AS DATE), CAST(hr AS INTEGER)) AS key_count "
    "FROM raw_data"
    ") "
    "SELECT *, "
    "CONCAT_WS(';', "
    "CASE WHEN dteday IS NOT NULL AND hr IS NOT NULL AND key_count > 1 THEN 'UQ01' END, "
    "CASE WHEN dteday IS NULL OR hr IS NULL OR season IS NULL OR weathersit IS NULL OR cnt IS NULL THEN 'CM01' END, "
    "CASE WHEN cnt IS NOT NULL AND cnt < 0 THEN 'RG01' END, "
    "CASE WHEN hr IS NOT NULL AND (hr < 0 OR hr > 23) THEN 'RG02' END, "
    "CASE WHEN casual IS NOT NULL AND registered IS NOT NULL AND cnt IS NOT NULL "
    "AND casual + registered <> cnt THEN 'CONS01' END"
    ") AS quality_rule_ids, "
    "CONCAT_WS(';', "
    "CASE WHEN season IS NOT NULL AND season NOT IN (1,2,3,4) THEN 'CAT01' END, "
    "CASE WHEN weathersit IS NOT NULL AND weathersit NOT IN (1,2,3,4) THEN 'CAT02' END"
    ") AS quality_flag_ids, "
    "CONCAT_WS('; ', "
    "CASE WHEN season IS NOT NULL AND season NOT IN (1,2,3,4) "
    "THEN 'недопустимый код season' END, "
    "CASE WHEN weathersit IS NOT NULL AND weathersit NOT IN (1,2,3,4) "
    "THEN 'недопустимый код weathersit' END"
    ") AS quality_flag_reason, "
    "CONCAT_WS('; ', "
    "CASE WHEN dteday IS NOT NULL AND hr IS NOT NULL AND key_count > 1 "
    "THEN 'дубликат dteday + hr' END, "
    "CASE WHEN dteday IS NULL OR hr IS NULL OR season IS NULL OR weathersit IS NULL OR cnt IS NULL "
    "THEN 'пропуск значимого поля' END, "
    "CASE WHEN cnt IS NOT NULL AND cnt < 0 THEN 'отрицательное cnt' END, "
    "CASE WHEN hr IS NOT NULL AND (hr < 0 OR hr > 23) "
    "THEN 'hr вне диапазона 0..23' END, "
    "CASE WHEN casual IS NOT NULL AND registered IS NOT NULL AND cnt IS NOT NULL "
    "AND casual + registered <> cnt THEN 'casual + registered != cnt' END"
    ") AS quarantine_reason "
    "FROM base"
)

con.execute(
    f"COPY ("
    f"SELECT instant, dteday, season, yr, mnth, hr, holiday, weekday, workingday, "
    f"weathersit, temp, atemp, hum, windspeed, casual, registered, cnt, "
    f"quality_flag_ids, quality_flag_reason "
    f"FROM normalized_with_quality "
    f"WHERE quality_rule_ids = ''"
    f") TO '{STAGING_PATH}' "
    f"(FORMAT PARQUET, OVERWRITE_OR_IGNORE TRUE)"
)

con.execute(
    f"COPY ("
    f"SELECT instant, dteday, season, yr, mnth, hr, holiday, weekday, workingday, "
    f"weathersit, temp, atemp, hum, windspeed, casual, registered, cnt, "
    f"quality_rule_ids, quarantine_reason "
    f"FROM normalized_with_quality "
    f"WHERE quality_rule_ids <> ''"
    f") TO '{QUARANTINE_PATH}' "
    f"(FORMAT PARQUET, OVERWRITE_OR_IGNORE TRUE)"
)

print("Parquet-файлы записаны.")
print(STAGING_PATH)
print(QUARANTINE_PATH)


Parquet-файлы записаны.
s3://staging/bike-sharing/source_ingested_on=2026-10-18/staging.parquet
s3://staging/bike-sharing/source_ingested_on=2026-10-18/quarantine.parquet


## 4. Проверка подготовленных данных

Проверяем баланс `raw = staging + quarantine`, отсутствие критических нарушений в `staging`, причины в карантине и сохранение предупреждающих флагов.


In [19]:
raw_count = int(
    con.execute("SELECT COUNT(*) FROM raw_data").fetchone()[0]
)

staging_count = int(
    con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{STAGING_PATH}')"
    ).fetchone()[0]
)

quarantine_count = int(
    con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{QUARANTINE_PATH}')"
    ).fetchone()[0]
)

pd.DataFrame([{
    "raw_rows": raw_count,
    "staging_rows": staging_count,
    "quarantine_rows": quarantine_count,
    "staging_plus_quarantine": staging_count + quarantine_count,
    "balance_ok": raw_count == staging_count + quarantine_count,
}])


,raw_rows,staging_rows,quarantine_rows,staging_plus_quarantine,balance_ok
0,17379,17379,0,17379,True


In [20]:
critical_on_staging = con.execute(
    f"SELECT "
    f"COUNT(*) FILTER (WHERE cnt < 0) AS rg01_violations, "
    f"COUNT(*) FILTER (WHERE hr < 0 OR hr > 23) AS rg02_violations, "
    f"COUNT(*) FILTER ("
    f"WHERE dteday IS NULL OR hr IS NULL OR season IS NULL "
    f"OR weathersit IS NULL OR cnt IS NULL"
    f") AS cm01_violations, "
    f"COUNT(*) FILTER ("
    f"WHERE casual IS NOT NULL AND registered IS NOT NULL AND cnt IS NOT NULL "
    f"AND casual + registered <> cnt"
    f") AS cons01_violations "
    f"FROM read_parquet('{STAGING_PATH}')"
).df()

critical_on_staging


,rg01_violations,rg02_violations,cm01_violations,cons01_violations
0,0,0,0,0


In [21]:
quarantine_check = con.execute(
    f"SELECT "
    f"COUNT(*) AS quarantine_rows, "
    f"COUNT(*) FILTER (WHERE quality_rule_ids = '') AS rows_without_rule_id, "
    f"COUNT(*) FILTER (WHERE quarantine_reason = '') AS rows_without_reason "
    f"FROM read_parquet('{QUARANTINE_PATH}')"
).df()

quarantine_check


,quarantine_rows,rows_without_rule_id,rows_without_reason
0,0,0,0


In [22]:
flag_check = con.execute(
    f"SELECT "
    f"COUNT(*) FILTER (WHERE quality_flag_ids LIKE '%CAT01%') AS cat01_flags, "
    f"COUNT(*) FILTER (WHERE quality_flag_ids LIKE '%CAT02%') AS cat02_flags, "
    f"COUNT(*) FILTER (WHERE season IS NOT NULL AND season NOT IN (1,2,3,4)) AS cat01_rows, "
    f"COUNT(*) FILTER (WHERE weathersit IS NOT NULL AND weathersit NOT IN (1,2,3,4)) AS cat02_rows "
    f"FROM read_parquet('{STAGING_PATH}')"
).df()

flag_check


,cat01_flags,cat02_flags,cat01_rows,cat02_rows
0,0,0,0,0


## 5. Воспроизводимость

После первого полного выполнения зафиксировать в `docs/data_quality_report.md` количества строк `raw`, `staging` и `quarantine`, а также результаты всех правил.

Затем перезапустить ядро Jupyter и выполнить все ячейки сверху вниз с теми же входными данными, порогами и выходным префиксом. Количества строк двух результатов должны совпасть с первым выполнением.
